# TP MLOps II — Baseline de forecasting

Continuación de la limpieza (`02_dataset_cleaning.ipynb`). Objetivo: armar un **split temporal** train/test y un **primer modelo baseline** para predecir `total load actual` (demanda eléctrica horaria de España), evitando data leakage y comparando contra el forecast oficial del operador de red (`total load forecast`) como benchmark de industria.

## 1. Carga y split temporal

Se carga el dataset limpio (`data/processed/energy_weather_clean.parquet`) y se separa en train/test **por fecha, no al azar** — así se simula el escenario real de predecir el futuro con datos del pasado, y se evita que el modelo "vea" información de fechas posteriores durante el entrenamiento.

**Corte elegido:** train = 2015-2017 (3 años), test = 2018 completo. Usar el último año entero como test permite evaluar el modelo en las 4 estaciones del año, en vez de un tramo corto que podría sesgar el resultado a un solo régimen climático.

In [4]:
from pathlib import Path

import pandas as pd

data_path = Path("../data")
df = pd.read_parquet(data_path / "processed" / "energy_weather_clean.parquet")

train = df[df.index.year < 2018]
test = df[df.index.year >= 2018]

print(train.shape, test.shape)
print(train.index.min(), train.index.max())
print(test.index.min(), test.index.max())

(26305, 35) (8759, 35)
2014-12-31 23:00:00+00:00 2017-12-31 23:00:00+00:00
2018-01-01 00:00:00+00:00 2018-12-31 22:00:00+00:00


## 2. Selección de features — evitando data leakage

El dataset limpio tiene columnas que en un escenario real **no estarían disponibles en el momento de predecir**: generación real por fuente, precio real, etc. son valores medidos en la misma hora que se quiere predecir, no datos conocidos de antemano.

Para un baseline honesto (pensando en que más adelante esto se convierta en un modelo servido en producción), se usan solo:
- **Calendario:** `hour`, `dow`, `month`, `is_weekend`.
- **Clima:** temperatura por ciudad (`temp_<ciudad>`), que si se usa el forecast de clima en vez del valor real, también estaría disponible con anticipación.
- **`total load forecast`:** el forecast oficial del operador de red — sí está disponible de antemano (se publica día-adelantado), por eso es válido usarlo como feature de entrada.

Se descartan todas las columnas de generación real y precios reales.

In [5]:
feature_cols = ['hour', 'dow', 'month', 'is_weekend',
                 'temp_Madrid', 'temp_Barcelona', 'temp_Valencia', 'temp_Seville', 'temp_Bilbao',
                 'total load forecast']
target_col = 'total load actual'

X_train, y_train = train[feature_cols], train[target_col]
X_test, y_test = test[feature_cols], test[target_col]

X_train.shape, X_test.shape

((26305, 10), (8759, 10))

## 3. Modelo baseline: regresión lineal

Se arranca con el modelo más simple posible (`LinearRegression` de scikit-learn) para establecer un piso honesto de referencia, antes de probar modelos más complejos. Al ser lineal, no captura del todo relaciones no lineales como la forma en U entre temperatura y demanda (vista en el EDA) ni la naturaleza cíclica de `hour`/`month` — pero sirve como punto de partida interpretable.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

## 4. Métricas

Se evalúa con **MAE** (error absoluto promedio en MW, fácil de interpretar en la escala del problema) y **MAPE** (error porcentual, comparable entre distintas magnitudes de demanda), además de RMSE.

In [ ]:
from sklearn.metrics import (
                 mean_absolute_error,
                 mean_absolute_percentage_error,
                 root_mean_squared_error,
)

mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print(f"MAE:   {mae:.1f} MW")
print(f"MAPE: {mape:.2%}")
print(f"RMSE:  {rmse:.1f} MW")

0.009316508836797946
MAE:   271.8 MW
MAPE: 0.93%
RMSE:  390.6 MW


## 5. Comparación contra el benchmark de industria

El dataset trae `total load forecast`: el forecast oficial que publicó el operador de red español para cada una de esas horas. Es la vara de referencia real de la industria — comparar contra esto (y no solo contra "cero conocimiento") es mucho más informativo que un baseline arbitrario.

In [9]:
mae_benchmark = mean_absolute_error(y_test, test['total load forecast'])
mape_benchmark = mean_absolute_percentage_error(y_test, test['total load forecast'])

print(f"Benchmark TSO — MAE: {mae_benchmark:.1f} MW | MAPE: {mape_benchmark:.2%}")


Benchmark TSO — MAE: 269.9 MW | MAPE: 0.93%


## Resultado

| Modelo | MAE | MAPE |
|---|---|---|
| Regresión lineal (baseline propio) | 271.8 MW | 0.93% |
| Forecast oficial del TSO (benchmark) | 269.9 MW | 0.93% |

**El baseline lineal prácticamente empata al forecast oficial del operador de red**, con un conjunto mínimo de features (calendario + temperatura + el forecast del TSO como input) y el modelo más simple posible. Es un resultado muy fuerte para un primer intento — el techo de mejora respecto al benchmark real de industria es chico, lo cual también es información valiosa: el problema ya está bastante bien resuelto por el forecast oficial, y superarlo de forma consistente será difícil.

**Próximos pasos posibles:** feature engineering más rico (lags, ventanas móviles, variables cíclicas para hour/month), o probar modelos no lineales (Random Forest, Gradient Boosting) para ver si capturan la relación en U de la temperatura y mejoran más allá del benchmark — y en paralelo, empezar a pensar la infraestructura para servir este modelo (Sesión 1: API REST).